# Creating Dataset

In [66]:
import pandas as pd
import random

random.seed(42)

campaigns = [
    "Facebook",
    "Google Ads",
    "Instagram",
    "TikTok",
    "YouTube"
]

dates = pd.date_range(
    start="2020-11-01",
    periods=100,
    freq="D"
)

data = []

for i, date in enumerate(dates):

    campaign = random.choice(campaigns)

    budget = random.randint(4000, 10000)
    clicks = random.randint(900, 2500)
    conversion = round(random.uniform(3.5, 8.0), 1)
    revenue = random.randint(18000, 50000)

    # Different date formats
    if i % 15 == 0:
        date_value = date.strftime("%B %d, %Y")
    elif i % 20 == 0:
        date_value = date.strftime("%Y/%m/%d")
    else:
        date_value = date.strftime("%Y-%m-%d")

    # Budget with special characters
    if i % 3 == 0:
        budget_value = f"₱{budget:,}"
    elif i % 3 == 1:
        budget_value = f"{budget}"
    else:
        budget_value = f"₱{budget}"

    # Clicks as strings with commas
    if i % 4 == 0:
        clicks_value = f"{clicks:,}"
    else:
        clicks_value = str(clicks)

    # Conversion rate with %
    conversion_value = f"{conversion}%"

    # Revenue with currency symbol
    if i % 2 == 0:
        revenue_value = f"₱{revenue:,}"
    else:
        revenue_value = str(revenue)

    data.append([
        date_value,
        campaign,
        budget_value,
        clicks_value,
        conversion_value,
        revenue_value
    ])

# Convert to DataFrame
df = pd.DataFrame(
    data,
    columns=[
        "Date",
        "Campaign",
        "Budget",
        "Clicks",
        "Conversion_Rate",
        "Revenue"
    ]
)

# INSERT NULL VALUES
df.loc[5, "Budget"] = None
df.loc[12, "Clicks"] = None
df.loc[20, "Conversion_Rate"] = None
df.loc[27, "Revenue"] = None
df.loc[35, "Campaign"] = None
df.loc[44, "Date"] = None
df.loc[53, "Budget"] = None
df.loc[66, "Clicks"] = None
df.loc[78, "Revenue"] = None
df.loc[91, "Conversion_Rate"] = None

# INSERT OUTLIERS
df.loc[10, "Budget"] = "₱500,000"
df.loc[25, "Clicks"] = "95,000"
df.loc[40, "Conversion_Rate"] = "85%"
df.loc[60, "Revenue"] = "₱5,000,000"

df.loc[75, "Budget"] = "₱250,000"
df.loc[85, "Clicks"] = "50,000"

# INSERT INCONSISTENT CAMPAIGN NAMES
df.loc[15, "Campaign"] = "facebook"
df.loc[30, "Campaign"] = "FACEBOOK"
df.loc[45, "Campaign"] = "Google ads"
df.loc[70, "Campaign"] = "instagram"
df.loc[95, "Campaign"] = "Tik Tok"

# SAVE CSV
df.to_csv("Marketing_Messy_100.csv", index=False)

print("Marketing_Messy_100.csv created successfully!")
print("Number of records:", len(df))

Marketing_Messy_100.csv created successfully!
Number of records: 100


# Data Error Checker

- **Null Values**
- **Data Type Mismatch**
- **Special Characters**
- **Outliers**


In [67]:
def check_data(df):

    results = []

   
    # NULL VALUES
    null_count = df.isnull().sum().sum()

    if null_count == 0:
        results.append([
            "Null Values",
            "PASS",
            "No null values found"
        ])
    else:
        results.append([
            "Null Values",
            "FAIL",
            f"{null_count} null value(s) found"
        ])


    # DATA TYPES
    datatype_errors = []

    # Date
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        datatype_errors.append(
            f"Date: {df['Date'].dtype}"
        )

    # Campaign
    if not pd.api.types.is_string_dtype(df["Campaign"]):
        datatype_errors.append(
            f"Campaign: {df['Campaign'].dtype}"
        )

    # Numeric columns
    numeric_columns = [
        "Budget",
        "Clicks",
        "Conversion_Rate",
        "Revenue"
    ]

    for column in numeric_columns:

        if not pd.api.types.is_numeric_dtype(df[column]):
            datatype_errors.append(
                f"{column}: {df[column].dtype}"
            )


    if len(datatype_errors) == 0:
        results.append([
            "Data Types",
            "PASS",
            "All columns have appropriate data types"
        ])
    else:
        results.append([
            "Data Types",
            "FAIL",
            ", ".join(datatype_errors)
        ])


    
    # SPECIAL CHARACTERS
    

    special_found = []

    checks = {
        "Budget": ["₱", ","],
        "Clicks": [","],
        "Conversion_Rate": ["%"],
        "Revenue": ["₱", ","]
    }

    for column, characters in checks.items():

        for char in characters:

            found = (
                df[column]
                .astype("string")
                .str.contains(
                    char,
                    regex=False,
                    na=False
                )
                .any()
            )

            if found:
                special_found.append(
                    f"{column} contains {char}"
                )


    if len(special_found) == 0:
        results.append([
            "Special Characters",
            "PASS",
            "No unwanted special characters"
        ])
    else:
        results.append([
            "Special Characters",
            "FAIL",
            ", ".join(special_found)
        ])


    
    # DATE VALIDITY
    invalid_dates = df["Date"].isna().sum()

    if invalid_dates == 0:
        results.append([
            "Date Values",
            "PASS",
            "All dates are valid"
        ])
    else:
        results.append([
            "Date Values",
            "FAIL",
            f"{invalid_dates} invalid/missing date(s)"
        ])


    # CAMPAIGN NAMES
    allowed_campaigns = {
        "Facebook",
        "Google Ads",
        "Instagram",
        "TikTok",
        "YouTube",
        "Unknown"
    }

    invalid_campaigns = (
        set(df["Campaign"].dropna().unique())
        - allowed_campaigns
    )

    if len(invalid_campaigns) == 0:
        results.append([
            "Campaign Names",
            "PASS",
            "Campaign names are standardized"
        ])
    else:
        results.append([
            "Campaign Names",
            "FAIL",
            f"Invalid values: {invalid_campaigns}"
        ])

    # CHECK OUTLIERS USING IQR
    numeric_columns = [
        "Budget",
        "Clicks",
        "Conversion_Rate",
        "Revenue"
    ]

    outlier_details = []

    for column in numeric_columns:

        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)

        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        outliers = df[
            (df[column] < lower_limit) |
            (df[column] > upper_limit)
        ]

        if len(outliers) > 0:

            outlier_details.append(
                f"{column}: {len(outliers)} outlier(s)"
            )


    if len(outlier_details) == 0:

        results.append([
            "Outliers",
            "PASS",
            "No outliers detected"
        ])

    else:

        results.append([
            "Outliers",
            "FAIL",
            ", ".join(outlier_details)
        ])


    
    # DISPLAY RESULTS
    results_df = pd.DataFrame(
        results,
        columns=[
            "Check",
            "Status",
            "Details"
        ]
    )

    display(results_df)


    
    # OVERALL RESULT
    if (results_df["Status"] == "PASS").all():

        print("\nOVERALL RESULT: PASS")
        print("All data cleaning checks passed. You can export the cleaned dataset.")

    else:

        print("\nOVERALL RESULT: FAIL")
        print("Some data cleaning checks failed.")

# Display Total Records

In [68]:
pd.set_option("display.max_rows", None)
df

,Date,Campaign,Budget,Clicks,Conversion_Rate,Revenue
0,"November 01, 2020",Facebook,"₱4,204","2,418",4.7%,"₱25,314"
1,2020-11-02,Google Ads,4839,2285,6.8%,35870
2,2020-11-03,Facebook,₱8837,1764,3.6%,"₱21,070"
3,2020-11-04,Google Ads,"₱5,905",1934,6.2%,36390
4,2020-11-05,Google Ads,9865,"2,230",6.7%,"₱31,746"
5,2020-11-06,Google Ads,NaN,2106,4.8%,46485
6,2020-11-07,Facebook,"₱5,307",2329,5.4%,"₱27,105"
7,2020-11-08,Google Ads,5763,2463,5.0%,21039
8,2020-11-09,TikTok,₱4792,"1,635",7.3%,"₱37,782"
9,2020-11-10,Instagram,"₱4,355",2394,5.6%,22090


# Check Special Characters

In [69]:
df["Budget"].head(10)
df.head()

,Date,Campaign,Budget,Clicks,Conversion_Rate,Revenue
0,"November 01, 2020",Facebook,"₱4,204","2,418",4.7%,"₱25,314"
1,2020-11-02,Google Ads,4839,2285,6.8%,35870
2,2020-11-03,Facebook,₱8837,1764,3.6%,"₱21,070"
3,2020-11-04,Google Ads,"₱5,905",1934,6.2%,36390
4,2020-11-05,Google Ads,9865,"2,230",6.7%,"₱31,746"


# Checking Data Type Mismatch

In [70]:
df.dtypes

Date               str
Campaign           str
Budget             str
Clicks             str
Conversion_Rate    str
Revenue            str
dtype: object

# Convert Date from str to datetime

In [71]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df.dtypes

Date               datetime64[us]
Campaign                      str
Budget                        str
Clicks                        str
Conversion_Rate               str
Revenue                       str
dtype: object

# Clean the special characters (Budget)

In [72]:
df["Budget"] = (
    df["Budget"]
    .str.replace("₱", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["Budget"] = pd.to_numeric(df["Budget"], errors="coerce")

# Clean the special characters (Conversion_Rate)

In [73]:
df["Conversion_Rate"] = (
    df["Conversion_Rate"]
    .str.replace("%", "", regex=False)
)

df["Conversion_Rate"] = pd.to_numeric(
    df["Conversion_Rate"],
    errors="coerce"
)

# Clean the special characters (Revenue)

In [74]:
df["Revenue"] = (
    df["Revenue"]
    .str.replace("₱", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["Revenue"] = pd.to_numeric(
    df["Revenue"],
    errors="coerce"
) 

# Clean the special characters (Click)

In [75]:
df["Clicks"] = (
    df["Clicks"]
    .str.replace(",", "", regex=False)
)

df["Clicks"] = pd.to_numeric(
    df["Clicks"],
    errors="coerce"
)

# Fill in the missing values (numeric)

In [76]:
df["Budget"] = df["Budget"].fillna(
    df["Budget"].median()
)

df["Clicks"] = df["Clicks"].fillna(
    df["Clicks"].median()
)

df["Conversion_Rate"] = df["Conversion_Rate"].fillna(
    df["Conversion_Rate"].median()
)

df["Revenue"] = df["Revenue"].fillna(
    df["Revenue"].median()
)

# Fill in missing values (Campaign)

In [77]:
df["Campaign"] = df["Campaign"].fillna("Unknown")

# Clean Inconsistent Campaign Names

In [84]:
df["Campaign"] = df["Campaign"].replace({
    "Tik Tok": "TikTok",
    "google ads": "Google Ads",
    "Google ads": "Google Ads",
    "FACEBOOK": "Facebook",
    "facebook": "Facebook",
    "instagram": "Instagram"
})

# Replace NaN with 0 (numeric)

In [79]:
numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = df[numeric_columns].fillna(0)

# Replace NaT (Not a Time) within existing date range

In [80]:
import random
min_date = df["Date"].min()
max_date = df["Date"].max()
for index in df[df["Date"].isna()].index:
    random_days = random.randint(
        0,
        (max_date - min_date).days
        )
    df.loc[index, "Date"] = (
        min_date + pd.Timedelta(days=random_days)
    )

# Check for the outliers

In [81]:
def show_outliers(df):

    numeric_columns = [
        "Budget",
        "Clicks",
        "Conversion_Rate",
        "Revenue"
    ]

    for column in numeric_columns:

        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)

        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

    outliers = df[
                (df[column] < lower_limit) |
                (df[column] > upper_limit)
            ]

    if len(outliers) > 0:

        print(f"\n===== {column} OUTLIERS =====")

        display(
            outliers[
                ["Date", "Campaign", column]
                ]
            )


# Solve for the outliers

In [86]:
numeric_columns = [
    "Budget",
    "Clicks",
    "Conversion_Rate",
    "Revenue"
]

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    median = df[column].median()

    df.loc[
            (df[column] < lower_limit) |
            (df[column] > upper_limit),
            column
        ] = median

    Q1 = df["Revenue"].quantile(0.25)
    Q3 = df["Revenue"].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    median = df["Revenue"].median()

    df.loc[
        (df["Revenue"] < lower_limit) |
        (df["Revenue"] > upper_limit),
        "Revenue"
    ] = median

# Check data

In [87]:
check_data(df)

,Check,Status,Details
0,Null Values,PASS,No null values found
1,Data Types,PASS,All columns have appropriate data types
2,Special Characters,PASS,No unwanted special characters
3,Date Values,PASS,All dates are valid
4,Campaign Names,PASS,Campaign names are standardized
5,Outliers,PASS,No outliers detected



OVERALL RESULT: PASS
All data cleaning checks passed. You can export the cleaned dataset.


# Export cleaned data to CSV file

In [88]:
import os

df.to_csv("Marketing_Cleaned.csv", index=False)
print("Cleaned dataset exported successfully!")
print(os.path.exists("Marketing_Cleaned.csv"))

Cleaned dataset exported successfully!
True
